# Simulations moteur & correcteurs

## Imports

In [66]:
import numpy as np
from control import tf, c2d, feedback, damp
from control_plotly import pzmap, step, impulse, bode, nichols, rlocus

### Moteur en no-load

In [67]:
### mt_ = moteur
mt_L = 1.4e-3 #H
mt_R = 1.4 #Ohm
mt_Km = 57e-3 #Nm/A
mt_J = 2.5e-5 #Kg/m2
mt_f = 8e-4 #N.m/(rad/s)

s = tf('s')

Mt_s = (mt_Km)/((mt_Km**2)+(mt_f+mt_J*s)*(mt_R+mt_L*s))
print(Mt_s)


               0.057
------------------------------------
3.5e-08 s^2 + 3.612e-05 s + 0.004369



### Simulations

In [68]:
### Simulation moteur
Mt_s_rpm = (60/(2*np.pi))*Mt_s
tm = np.linspace(start=0,stop=0.1, num = 1000)
step(Mt_s_rpm, tm, x_title="Temps (s)", y_title ="RPM")

In [70]:
Mt_s_H = Mt_s_rpm/(1+Mt_s_rpm)
step([Mt_s_H*125,Mt_s_rpm],t=tm, x_title = "Temps (s)", y_title ="Commande")

## Discretisation en no-load

In [71]:
tm_z = np.arange(0, 1, 0.01)
Mt_z_rpm = c2d(Mt_s_rpm, 0.01, 'zoh')
print(Mt_z_rpm)
step([Mt_z_rpm,Mt_s_rpm], t=tm_z)


     88.12 z + 5.704
--------------------------
z^2 - 0.2469 z + 3.297e-05

dt = 0.01



### Boucle fermée discrete

In [72]:
Mt_z_H = Mt_z_rpm/(1+Mt_z_rpm)
step(Mt_z_H, t=tm_z)

In [73]:
offset = 10**(36.664/20) #+1 for non-oscillating graph
print(offset)
nichols([Mt_z_rpm,Mt_z_rpm/offset, Mt_z_rpm/80])

68.10829367676388


In [75]:
Mt_z_Hcorr = (Mt_z_rpm/(offset))/(1+(Mt_z_rpm/(offset)))
tm_z2 = np.arange(0, 1, 0.01)
step(Mt_z_Hcorr,t=tm_z2)

In [76]:
tm_z2 = np.arange(0, 5, 0.01)
step([Mt_z_Hcorr,Mt_s_H],t=tm_z2)

### Correcteur

In [77]:
Te = 0.01

Kp = 0.004#1/(offset + 120)
Ki = 0.008
Kd = -0.0001#0.09

z = tf('z')

print("Kp : ", Kp)
print("Ki : ", Ki)
print("Kd : ", Kd)

Cz = Kp+Ki*(z/(z-1))+Kd*((z-1)/z)

nichols([Cz*Mt_z_rpm, (Cz*Mt_z_rpm)/(1+Cz*Mt_z_rpm)])

Kp :  0.004
Ki :  0.008
Kd :  -0.0001


In [81]:
Mt_z_corr = (Cz*Mt_z_rpm)/(1+(Cz*Mt_z_rpm))
tsim = np.arange(0,0.5,Te)
step(1*Mt_z_corr, tsim)

In [82]:
Mt_z_errcorr = 1/(1+(Cz*Mt_z_rpm))
step(Mt_z_errcorr*3000, tsim)

In [86]:
# =============
# Commande
# =============
cmd_z_rpm = Cz/(1+(Cz*Mt_z_rpm))
cmd_z_rpm /= (Ki+Kd+Kp)

step(100*cmd_z_rpm, t =np.arange(0,1,0.01), x_title="Temps (s)", y_title="Commande PWM (%)")

## Moteur chargé

In [3]:
# =======================================
#   Calcul moment d'inertie du volant
# =======================================


def Iv(R, r, m):
    result = 0.5*m*(r+R)**2
    return result

def IvAtMotor(R,r, m, reduction):
    result = (1/reduction**2)*Iv(R,r,m)
    return result

r_volant = 23e-3 #m
R_volant = 148e-3 #m
m = 2.3 #kg
reduction = 15.88 #tr_e / tr_s


IvMotor = IvAtMotor(R_volant, r_volant, m, reduction)
print(IvMotor)

0.00013334878560234502


In [7]:
# =======================================
#   Equation moteur
# =======================================
mt_L = 1.4e-3 #H
mt_R = 1.4 #Ohm
mt_Km = 57e-3 #Nm/A
mt_J = 2.5e-5 + IvMotor#Kg/m2
mt_f = 8e-4 #N.m/(rad/s)

s = tf('s')

Mt_s_loaded = (mt_Km)/((mt_Km**2)+(mt_f+mt_J*s)*(mt_R+mt_L*s))
print(Mt_s_loaded)


                0.057
--------------------------------------
2.217e-07 s^2 + 0.0002228 s + 0.004369



In [16]:
### Simulation moteur
Mt_s_loaded_rpm = (60/(2*np.pi))*Mt_s_loaded
tm = np.linspace(start=0,stop=0.3, num = 2000)
step([Mt_s_rpm, Mt_s_loaded_rpm], tm, x_title="Temps (s)", y_title ="RPM")

In [90]:
# =========================================
#  Simulation retard vis-à-vis du no-load
# =========================================

tm = np.linspace(start=0,stop=0.4, num = 2000)
step((Mt_s_rpm - Mt_s_loaded_rpm), tm, x_title="Temps (s)", y_title ="RPM")

In [18]:
### Simulation moteur
Mt_h_loaded_rpm = Mt_s_loaded_rpm/(1+Mt_s_loaded_rpm)
tm = np.linspace(start=0,stop=0.3, num = 2000)
step([Mt_s_loaded_rpm, Mt_h_loaded_rpm*125], tm, x_title="Temps (s)", y_title ="RPM")

### Discret

In [19]:
tm_z = np.arange(0, 1, 0.01)
Mt_z_loaded_rpm = c2d(Mt_s_loaded_rpm, 0.01, 'zoh')
print(Mt_z_loaded_rpm)
step([Mt_z_loaded_rpm,Mt_s_loaded_rpm], t=tm_z)


     20.48 z + 2.113
--------------------------
z^2 - 0.8187 z + 4.316e-05

dt = 0.01



In [93]:
# =========================================
#  Simulation retard vis-à-vis du no-load
# =========================================

tm = np.linspace(start=0,stop=0.4, num = 2000)
step((Mt_z_rpm - Mt_z_loaded_rpm), t=np.arange(0, 0.5, 0.01), x_title="Temps (s)", y_title ="RPM")

### Visualisation

In [24]:
# ======================================
#  Nichols moteur brut + offset de base
# ======================================

offset = 10**(20.1/20)
nichols([Mt_z_loaded_rpm, Mt_z_loaded_rpm/offset])

### Correcteur discret chargé

In [62]:
# ====================
#  Simulation moteur
# ====================

Te = 0.01

Kp_l = 1/(offset + 30)
Ki_l = 0.0055
Kd_l = 0.001

z = tf('z')

print("Kp : ", Kp_l)
print("Ki : ", Ki_l)
print("Kd : ", Kd_l)

Cz_loaded = Kp_l+Ki_l*(z/(z-1))+Kd_l*((z-1)/z)

HBO_z_loaded = Cz_loaded*Mt_z_loaded_rpm
HBF_z_loaded = (Cz_loaded*Mt_z_loaded_rpm)/(1+Cz_loaded*Mt_z_loaded_rpm)

nichols([HBO_z_loaded, HBF_z_loaded])

Kp :  0.024927837312011838
Ki :  0.0055
Kd :  0.001


In [63]:
# =============
# step response
# =============
step([HBF_z_loaded] ,t=tm_z)

In [64]:
# =============
# Commande
# =============
Cmd_z_loaded = Cz_loaded/(1+HBO_z_loaded)

step([100*Cmd_z_loaded/(Kp_l+Ki_l+Kd_l)] ,t=tm_z, x_title = "Temps (s)", y_title = "Duty Cycle %")

In [65]:
# =============
# Commande (Cz)
# =============

impulse(Cz_loaded ,t=tm_z, x_title = "Temps (s)", y_title = "Voltage")